In [17]:
import pickle
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch_geometric.data import Data
from torch_geometric.nn import (
    GCNConv,
    SAGEConv,
    GATConv
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [18]:
SEED = 42

HIDDEN_DIM = 128
DROPOUT = 0.2

LEARNING_RATE = 0.005
WEIGHT_DECAY = 1e-4

EPOCHS = 300
PATIENCE = 30

PROJECT_ROOT = ".."

GRAPH_PATH = (
    "../data/processed/graph_data.pkl"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)

Device: mps


In [19]:
with open(GRAPH_PATH, "rb") as f:
    graph = pickle.load(f)

nodes = graph["nodes"]

edge_index_np = graph["edge_index"]

train_mask_np = graph["train_mask"]
val_mask_np = graph["val_mask"]
test_mask_np = graph["test_mask"]

print("Nodes:", len(nodes))
print("Edges:", edge_index_np.shape[1])

Nodes: 203769
Edges: 468710


In [20]:
feature_columns = [
    column
    for column in nodes.columns
    if column.startswith("feature_")
]

X = nodes[
    feature_columns
].values.astype(np.float32)

y = nodes[
    "label"
].values.astype(np.int64)

print("Features:", X.shape)
print("Labels:", y.shape)
print("Number of features:", len(feature_columns))

Features: (203769, 165)
Labels: (203769,)
Number of features: 165


In [21]:
x = torch.tensor(
    X,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    y,
    dtype=torch.long
)

edge_index = torch.tensor(
    edge_index_np,
    dtype=torch.long
)

train_mask = torch.tensor(
    train_mask_np,
    dtype=torch.bool
)

val_mask = torch.tensor(
    val_mask_np,
    dtype=torch.bool
)

test_mask = torch.tensor(
    test_mask_np,
    dtype=torch.bool
)

data = Data(
    x=x,
    edge_index=edge_index,
    y=y_tensor
)

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

data = data.to(DEVICE)

print(data)

Data(x=[203769, 165], edge_index=[2, 468710], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])


In [22]:
train_labels = y[train_mask_np]

negative_count = np.sum(
    train_labels == 0
)

positive_count = np.sum(
    train_labels == 1
)

pos_weight = (
    negative_count /
    positive_count
)

print("Negative:", negative_count)
print("Positive:", positive_count)
print("Positive weight:", pos_weight)

Negative: 26432
Positive: 3462
Positive weight: 7.634893125361063


In [23]:
class GCN(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        dropout
    ):

        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = GCNConv(
            hidden_dim,
            2
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.dropout(x)

        x = self.conv2(
            x,
            edge_index
        )

        return x


class GraphSAGE(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        dropout
    ):

        super().__init__()

        self.conv1 = SAGEConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = SAGEConv(
            hidden_dim,
            2
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.dropout(x)

        x = self.conv2(
            x,
            edge_index
        )

        return x


class GAT(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        dropout
    ):

        super().__init__()

        self.conv1 = GATConv(
            input_dim,
            hidden_dim,
            heads=1
        )

        self.conv2 = GATConv(
            hidden_dim,
            2,
            heads=1
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.dropout(x)

        x = self.conv2(
            x,
            edge_index
        )

        return x

In [29]:
def train_model(model, model_name):

    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(
            [1.0, pos_weight],
            dtype=torch.float32,
            device=DEVICE
        )
    )

    best_val_pr_auc = -np.inf
    best_state = None
    best_epoch = 0
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        optimizer.zero_grad()

        logits = model(
            data.x,
            data.edge_index
        )

        loss = criterion(
            logits[data.train_mask],
            data.y[data.train_mask]
        )

        loss.backward()

        optimizer.step()

        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        with torch.no_grad():

            logits = model(
                data.x,
                data.edge_index
            )

            val_logits = logits[data.val_mask]
            val_labels = data.y[data.val_mask]

            # Convert logits -> probability of class 1
            val_prob = torch.softmax(
                val_logits,
                dim=1
            )[:, 1]

            # Convert to NumPy
            y_val = (
                val_labels
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            p_val = (
                val_prob
                .detach()
                .cpu()
                .numpy()
                .astype(float)
            )

            # =================================================
            # PR-AUC
            # =================================================
            y_val = np.asarray(y_val).reshape(-1)
            p_val = np.asarray(p_val).reshape(-1)

            assert len(y_val) == len(p_val), (
                f"Length mismatch: y={len(y_val)}, p={len(p_val)}"
            )
            
            val_pr_auc = average_precision_score(
                y_val,
                p_val
            )

        # ====================================================
        # BEST MODEL
        # ====================================================

        if val_pr_auc > best_val_pr_auc:

            best_val_pr_auc = val_pr_auc

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            best_epoch = epoch

            patience_counter = 0

        else:

            patience_counter += 1

        # ====================================================
        # PROGRESS
        # ====================================================

        if epoch == 1 or epoch % 20 == 0:

            print(
                f"{model_name} | "
                f"Epoch {epoch:03d} | "
                f"Loss {loss.item():.4f} | "
                f"Val PR-AUC {val_pr_auc:.4f}"
            )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if patience_counter >= PATIENCE:

            print(
                f"\n{model_name} early stopped "
                f"at epoch {epoch}"
            )

            break

    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    model = model.to(DEVICE)

    training_time = (
        time.time() - start_time
    )

    print(
        "\n" + "=" * 60
    )

    print(
        f"{model_name} TRAINING COMPLETE"
    )

    print(
        f"Best epoch: {best_epoch}"
    )

    print(
        f"Best validation PR-AUC: "
        f"{best_val_pr_auc:.4f}"
    )

    print(
        f"Training time: "
        f"{training_time:.2f}s"
    )

    print(
        "=" * 60
    )

    return (
        model,
        best_epoch,
        best_val_pr_auc,
        training_time
    )

In [25]:
def evaluate_model(
    model,
    model_name,
    mask
):

    model.eval()

    with torch.no_grad():

        logits = model(
            data.x,
            data.edge_index
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

    probabilities = (
        probabilities
        .detach()
        .cpu()
        .numpy()
    )

    labels = (
        data.y[
            mask
        ]
        .detach()
        .cpu()
        .numpy()
    )

    probabilities = probabilities[
        mask.cpu().numpy()
    ]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    metrics = {

        "model": model_name,

        "pr_auc":
            average_precision_score(
                labels,
                probabilities
            ),

        "roc_auc":
            roc_auc_score(
                labels,
                probabilities
            ),

        "precision":
            precision_score(
                labels,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                predictions,
                zero_division=0
            )
    }

    return metrics

In [27]:
gcn = GCN(
    input_dim=len(feature_columns),
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
)

gcn, gcn_epoch, gcn_val_pr_auc, gcn_time = train_model(
    gcn,
    "GCN"
)

ValueError: Expected 2D array, got 1D array instead:
array=[0.7318083  0.81130457 0.80762076 ... 0.55149555 0.72317505 0.70904374].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.